# Topo-Evidential U-Mamba — Evaluation & Visualization Notebook
Chạy từng cell theo thứ tự. Cell **[0] Setup** phải chạy trước tất cả.

In [1]:
# ════════════════════════════════════════════════════
# [0] SETUP — chạy cell này đầu tiên
# ════════════════════════════════════════════════════
import sys, os
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')   # đổi sang 'inline' nếu dùng %matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings('ignore')

# ── Đường dẫn — chỉnh sửa cho phù hợp với máy bạn ──
PROJECT_ROOT   = Path('.')                      # thư mục gốc của project
CHECKPOINT     = 'outputs/checkpoints/best_model.pth'
ACDC_ROOT      = 'data/ACDC-dataset'
MNM1_ROOT      = 'data/M&M1'
MNM2_ROOT      = 'data/MnM2'
OUT_FIG        = Path('outputs/figures');  OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_EVAL       = Path('outputs/eval');     OUT_EVAL.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

from src.data.acdc            import collect_acdc_slices, SliceDataset
from src.data.mnm             import collect_mnm1_slices, DomainShiftDataset
from src.data.transforms      import get_val_transforms
from src.models.topo_evidential_umamba import build_model
from src.evaluation.metrics   import (
    compute_segmentation_metrics, expected_calibration_error,
    uncertainty_error_correlation, coverage_accuracy_curve, MetricAggregator
)
from src.evaluation.clinical_metrics import (
    compute_cardiac_volumes, bland_altman_stats,
    ClinicalMetricAggregator, stack_slices_to_volume
)
from src.topology.persistence import compute_persistence_diagram

# ── Config ──
NUM_CLASSES  = 4
CLASS_NAMES  = ['BG', 'RV', 'Myo', 'LV']
SPATIAL_SIZE = (224, 224)
BATCH_SIZE   = 8
ECE_BINS     = 15
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL_COLORS = np.array([[0,0,0],[255,100,100],[100,255,100],[100,100,255]], dtype=np.uint8)

print(f'Device : {DEVICE}')
print(f'Project: {PROJECT_ROOT.resolve()}')
print('Setup complete ✓')

ModuleNotFoundError: No module named 'einops'

In [ ]:
# ════════════════════════════════════════════════════
# [1] LOAD MODEL
# ════════════════════════════════════════════════════
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

model = build_model(cfg)
ckpt  = torch.load(CHECKPOINT, map_location='cpu')
model.load_state_dict(ckpt['model_state'])
model = model.to(DEVICE).eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Loaded  : {CHECKPOINT}')
print(f'Epoch   : {ckpt.get("epoch", "?")}  |  Best Dice: {ckpt.get("best_dice", 0):.4f}')
print(f'Params  : {n_params:,}')

In [ ]:
# ════════════════════════════════════════════════════
# [2] INFERENCE HELPER  (dùng lại ở nhiều cell)
# ════════════════════════════════════════════════════
@torch.no_grad()
def run_inference(dataloader, tag=''):
    """Returns dict with probs, segs, uncertainty, labels, metrics."""
    model.eval()
    agg = MetricAggregator(NUM_CLASSES)
    clinical_agg = ClinicalMetricAggregator()
    dice_records, all_probs, all_labels = [], [], []
    all_unc, all_correct, samples = [], [], []

    for batch in dataloader:
        imgs    = batch['image'].to(DEVICE)
        gt_segs = batch['label'].cpu().numpy()

        out      = model(imgs, return_projections=False)
        probs    = out['probs'].cpu().numpy()          # B K H W
        seg_maps = probs.argmax(axis=1)                # B H W
        unc_maps = out['uncertainty'].cpu().numpy().squeeze(1)  # B H W

        for i in range(len(imgs)):
            sp = batch['spacing'][i].numpy() if 'spacing' in batch else np.array([1.5,1.5,8.0])
            sp2d = (float(sp[0]), float(sp[1]))
            sp3d = (float(sp[0]), float(sp[1]), float(sp[2]) if len(sp)>2 else 8.0)

            m = compute_segmentation_metrics(seg_maps[i], gt_segs[i], NUM_CLASSES, sp2d)
            m['patient_id'] = batch.get('patient_id', ['?']*len(imgs))[i]
            m['phase']      = batch.get('phase', ['?']*len(imgs))[i]
            agg.update(m)
            dice_records.append({k: m.get(k,0) for k in ['dice_c1','dice_c2','dice_c3','dice_mean']})

            pid = m['patient_id']
            phase = m['phase']
            slc = int(batch.get('slice_idx', [-1]*len(imgs))[i]) if 'slice_idx' in batch else -1
            clinical_agg.add_slice(pid, phase, slc, seg_maps[i], gt_segs[i], sp3d)

            step = 4
            all_probs.append(probs[i].transpose(1,2,0)[::step,::step].reshape(-1,NUM_CLASSES))
            all_labels.append(gt_segs[i][::step,::step].reshape(-1))
            all_unc.append(unc_maps[i][::step,::step].reshape(-1))
            all_correct.append((seg_maps[i]==gt_segs[i])[::step,::step].reshape(-1))

            if len(samples) < 6:
                samples.append((imgs[i].cpu().numpy().squeeze(),
                                seg_maps[i], gt_segs[i], unc_maps[i],
                                m['patient_id']))

    return dict(
        summary      = agg.summary(),
        dice_records = dice_records,
        all_probs    = np.concatenate(all_probs),
        all_labels   = np.concatenate(all_labels),
        all_unc      = np.concatenate(all_unc),
        all_correct  = np.concatenate(all_correct).astype(bool),
        samples      = samples,
        clinical_agg = clinical_agg,
        agg          = agg,
    )

print('run_inference() helper ready ✓')

In [ ]:
# ════════════════════════════════════════════════════
# [3] ACDC TEST SET INFERENCE
# ════════════════════════════════════════════════════
val_tf = get_val_transforms(SPATIAL_SIZE)

acdc_slices = collect_acdc_slices(ACDC_ROOT, split='testing')
acdc_ds     = SliceDataset(acdc_slices, transforms=val_tf)
acdc_dl     = DataLoader(acdc_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f'ACDC test: {len(acdc_slices)} slices from {len(set(s["patient_id"] for s in acdc_slices))} patients')
acdc = run_inference(acdc_dl, tag='ACDC')

s = acdc['summary']
print(f"\nACDC Test Results:")
print(f"  Dice RV : {s.get('dice_c1_mean',0):.4f} ± {s.get('dice_c1_std',0):.4f}")
print(f"  Dice Myo: {s.get('dice_c2_mean',0):.4f} ± {s.get('dice_c2_std',0):.4f}")
print(f"  Dice LV : {s.get('dice_c3_mean',0):.4f} ± {s.get('dice_c3_std',0):.4f}")
print(f"  Dice Avg: {s.get('dice_mean_mean',0):.4f}")
print(f"  HD95    : {s.get('hd95_mean_mean',0):.2f} mm")

In [ ]:
# ════════════════════════════════════════════════════
# [4] PLOT — Training Curves (từ history CSV nếu có)
# ════════════════════════════════════════════════════
history_path = Path('outputs/eval/training_history.csv')

if history_path.exists():
    hist = pd.read_csv(history_path)
    epochs = hist['epoch'] if 'epoch' in hist.columns else range(len(hist))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    if 'train_loss' in hist.columns:
        axes[0].plot(epochs, hist['train_loss'], color='#E53935', lw=2)
        axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=.3)

    if 'val_dice_mean' in hist.columns:
        vd = hist['val_dice_mean'].dropna()
        ve = epochs[:len(vd)]
        axes[1].plot(ve, vd, color='#1976D2', marker='o', markersize=3, lw=2)
        best_idx = vd.idxmax()
        axes[1].axvline(epochs[best_idx], color='red', ls='--', alpha=.6,
                        label=f'Best: {vd[best_idx]:.4f}')
        axes[1].set_ylim(0,1); axes[1].legend()
        axes[1].set_title('Validation Dice Mean'); axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=.3)

    if 'lr' in hist.columns:
        axes[2].semilogy(epochs, hist['lr'], color='#388E3C', lw=2)
        axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=.3)

    plt.tight_layout()
    plt.savefig(OUT_FIG / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: training_curves.png')
else:
    print(f'training_history.csv not found at {history_path}')
    print('Tip: trong Trainer, thêm dòng sau cuối train():')
    print('  pd.DataFrame(history).to_csv("outputs/eval/training_history.csv", index=False)')

In [ ]:
# ════════════════════════════════════════════════════
# [5] PLOT — ACDC Dice Violin per Class
# ════════════════════════════════════════════════════
df_dice = pd.DataFrame(acdc['dice_records'])
df_dice.to_csv(OUT_EVAL / 'acdc_dice_records.csv', index=False)

long = df_dice[['dice_c1','dice_c2','dice_c3']].melt(var_name='Class', value_name='Dice')
long['Class'] = long['Class'].map({'dice_c1':'RV','dice_c2':'Myo','dice_c3':'LV'})

fig, ax = plt.subplots(figsize=(8,5))
sns.violinplot(data=long, x='Class', y='Dice',
               palette=['#2196F3','#4CAF50','#FF5722'],
               inner='box', cut=0, ax=ax)
ax.set_ylim(0, 1.05)
ax.axhline(0.9, color='grey', ls='--', alpha=.5, label='DSC=0.90')
ax.axhline(0.85, color='red', ls='--', alpha=.5, label='DSC=0.85 (clinical)')
ax.set_title('ACDC Test — Dice per Class', fontsize=13, fontweight='bold')
ax.set_ylabel('Dice Similarity Coefficient')
ax.legend()
ax.grid(axis='y', alpha=.3)
for i, cls in enumerate(['RV','Myo','LV']):
    med = long[long['Class']==cls]['Dice'].median()
    ax.text(i, med+0.02, f'{med:.3f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_FIG / 'acdc_dice_violin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: acdc_dice_violin.png')

In [ ]:
# ════════════════════════════════════════════════════
# [6] PLOT — ACDC Reliability Diagram (ECE)
# ════════════════════════════════════════════════════
ece, bin_accs, bin_confs, bin_counts = expected_calibration_error(
    acdc['all_probs'], acdc['all_labels'], n_bins=ECE_BINS
)
aurrc = uncertainty_error_correlation(acdc['all_unc'], acdc['all_correct'])

print(f'ACDC ECE  : {ece:.4f}')
print(f'ACDC AURRC: {aurrc:.4f}')

n_bins = ECE_BINS
x = np.arange(n_bins)

fig, ax = plt.subplots(figsize=(7,5))
ax.bar(x, bin_accs,  0.8, color='#1976D2', alpha=0.85, label='Accuracy')
ax.bar(x, bin_confs, 0.8, color='#E53935', alpha=0.30, label='Confidence')
ax.plot([0, n_bins-1], [0, 1], 'k--', alpha=.4, lw=1, label='Perfect')
ax.set_xlabel('Confidence Bin')
ax.set_ylabel('Accuracy / Confidence')
ax.set_title(f'ACDC — Reliability Diagram  (ECE = {ece:.4f})', fontsize=12, fontweight='bold')
ax.set_xticks(x[::3])
ax.set_xticklabels([f'{i/n_bins:.2f}' for i in range(n_bins)][::3])
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(OUT_FIG / 'acdc_reliability_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

pd.DataFrame({'ece':[ece],'aurrc':[aurrc]}).to_csv(OUT_EVAL/'acdc_calibration.csv', index=False)
print('Saved: acdc_reliability_diagram.png')

In [ ]:
# ════════════════════════════════════════════════════
# [7] PLOT — ACDC Coverage-Accuracy Curve
# ════════════════════════════════════════════════════
covs, accs = coverage_accuracy_curve(acdc['all_unc'], acdc['all_correct'])
baseline   = float(acdc['all_correct'].mean())

fig, ax = plt.subplots(figsize=(7,5))
ax.plot(covs*100, accs*100, color='#1976D2', lw=2, label='Uncertainty-filtered')
ax.axhline(baseline*100, color='grey', ls='--', lw=1.5,
           label=f'Baseline accuracy ({baseline*100:.1f}%)')
ax.fill_between(covs*100, baseline*100, accs*100,
                where=accs>baseline, alpha=.15, color='#1976D2', label='Gain region')
ax.set_xlabel('Coverage (%)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('ACDC — Coverage-Accuracy Curve\n'
             '(reject high-uncertainty pixels → accuracy ↑)', fontsize=11, fontweight='bold')
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(OUT_FIG / 'acdc_coverage_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: acdc_coverage_accuracy.png')

In [ ]:
# ════════════════════════════════════════════════════
# [8] PLOT — Uncertainty Maps (3 sample slices)
# ════════════════════════════════════════════════════
def seg_to_rgb(seg):
    h, w = seg.shape
    rgb  = np.zeros((h, w, 3), dtype=np.uint8)
    for c in range(4): rgb[seg==c] = LABEL_COLORS[c]
    return rgb

for idx, (img, pred, gt, unc, pid) in enumerate(acdc['samples'][:3]):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img, cmap='gray');                axes[0].set_title('Input Image')
    axes[1].imshow(seg_to_rgb(gt));                  axes[1].set_title('Ground Truth')
    axes[2].imshow(seg_to_rgb(pred));                axes[2].set_title('Prediction')
    im = axes[3].imshow(unc, cmap='hot', vmin=0, vmax=1)
    axes[3].set_title(f'Uncertainty  (mean={unc.mean():.3f})')
    plt.colorbar(im, ax=axes[3], fraction=.046, pad=.04)

    import matplotlib.patches as mpatches
    patches = [mpatches.Patch(color=LABEL_COLORS[i]/255, label=CLASS_NAMES[i]) for i in range(1,4)]
    axes[2].legend(handles=patches, loc='lower right', fontsize=7)

    for ax in axes: ax.axis('off')
    fig.suptitle(f'Patient: {pid}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_FIG / f'acdc_uncertainty_sample_{idx}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: acdc_uncertainty_sample_{idx}.png')

In [ ]:
# ════════════════════════════════════════════════════
# [9] PLOT — Persistence Diagram
# ════════════════════════════════════════════════════
sample_img = acdc_slices[0]['image'].squeeze()
pds = compute_persistence_diagram(sample_img, max_dim=1, threshold=0.02)

colors = ['#1976D2', '#E53935']
fig, ax = plt.subplots(figsize=(6,6))
max_val = 0.0
for dim, pd_pts in enumerate(pds[:2]):
    if not pd_pts: continue
    pts = np.array(pd_pts)
    ax.scatter(pts[:,0], pts[:,1], s=20, alpha=.7, color=colors[dim], label=f'H{dim}')
    max_val = max(max_val, pts.max())
max_val = max(max_val, 1.0)
ax.plot([0,max_val],[0,max_val],'k--',alpha=.4,lw=0.8,label='Diagonal')
ax.set_xlim(0, max_val*1.05); ax.set_ylim(0, max_val*1.05)
ax.set_xlabel('Birth'); ax.set_ylabel('Death')
ax.set_title('Persistence Diagram\n(H0 = connected components, H1 = loops)',
             fontsize=11, fontweight='bold')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(OUT_FIG / 'acdc_persistence_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: acdc_persistence_diagram.png')

In [ ]:
# ════════════════════════════════════════════════════
# [10] PLOT — Clinical Metrics + Bland-Altman
# ════════════════════════════════════════════════════
pred_clinical, gt_clinical = acdc['clinical_agg'].compute_all()

if pred_clinical and gt_clinical:
    pred_df = pd.DataFrame(pred_clinical).set_index('patient_id')
    gt_df   = pd.DataFrame(gt_clinical).set_index('patient_id')
    common  = pred_df.index.intersection(gt_df.index)
    pred_df = pred_df.loc[common]; gt_df = gt_df.loc[common]

    pd.concat([pred_df, gt_df], keys=['pred','gt']).to_csv(OUT_EVAL/'acdc_clinical.csv')

    metrics = [
        ('LV_EF_pct',  '%',   'LV Ejection Fraction'),
        ('LV_EDV_mL',  'mL',  'LV End-Diastolic Volume'),
        ('LV_ESV_mL',  'mL',  'LV End-Systolic Volume'),
        ('Myo_mass_g', 'g',   'Myocardial Mass'),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    for col_idx, (metric, unit, label) in enumerate(metrics):
        if metric not in pred_df.columns: continue
        p = pred_df[metric].values
        g = gt_df[metric].values

        # Scatter
        ax = axes[0, col_idx]
        ax.scatter(g, p, alpha=.7, color='#1976D2', s=30)
        lim = [min(g.min(),p.min())*.9, max(g.max(),p.max())*1.1]
        ax.plot(lim, lim, 'k--', alpha=.4)
        r = np.corrcoef(g, p)[0,1]
        ax.set_xlabel(f'GT ({unit})'); ax.set_ylabel(f'Pred ({unit})')
        ax.set_title(f'{label}\nr = {r:.3f}'); ax.grid(alpha=.3)

        # Bland-Altman
        ax = axes[1, col_idx]
        diff = p - g;  mean = (p+g)/2
        bias = diff.mean();  loa = 1.96*diff.std()
        ax.scatter(mean, diff, alpha=.7, color='#E53935', s=30)
        ax.axhline(bias,      color='#E53935', lw=1.5, label=f'Bias={bias:+.2f}')
        ax.axhline(bias+loa,  color='#FB8C00', ls='--', lw=1.5)
        ax.axhline(bias-loa,  color='#FB8C00', ls='--', lw=1.5,
                   label=f'LoA=±{loa:.2f}')
        ax.axhline(0, color='black', lw=.5, alpha=.5)
        ax.set_xlabel(f'Mean ({unit})'); ax.set_ylabel(f'Pred-GT ({unit})')
        ax.set_title(f'Bland-Altman'); ax.legend(fontsize=8); ax.grid(alpha=.3)

    fig.suptitle('Clinical Metrics: Prediction vs Ground Truth', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_FIG / 'acdc_clinical_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: acdc_clinical_summary.png')
else:
    print('Not enough data for clinical metrics (need ED + ES per patient)')

In [ ]:
# ════════════════════════════════════════════════════
# [11] M&Ms-1 INFERENCE
# ════════════════════════════════════════════════════
mnm1_slices = collect_mnm1_slices(MNM1_ROOT, splits=['Testing', 'Validation'])
mnm1_ds     = DomainShiftDataset(mnm1_slices, transforms=val_tf)
mnm1_dl     = DataLoader(mnm1_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f'M&Ms-1: {len(mnm1_slices)} slices, '
      f'{len(set(s["patient_id"] for s in mnm1_slices))} patients')
mnm1 = run_inference(mnm1_dl, tag='MnM1')

s = mnm1['summary']
print(f"\nM&Ms-1 Results:")
print(f"  Dice RV : {s.get('dice_c1_mean',0):.4f} ± {s.get('dice_c1_std',0):.4f}")
print(f"  Dice Myo: {s.get('dice_c2_mean',0):.4f} ± {s.get('dice_c2_std',0):.4f}")
print(f"  Dice LV : {s.get('dice_c3_mean',0):.4f} ± {s.get('dice_c3_std',0):.4f}")
print(f"  Dice Avg: {s.get('dice_mean_mean',0):.4f}")

In [ ]:
# ════════════════════════════════════════════════════
# [12] PLOT — M&Ms-1 Domain Invariance per Vendor
# ════════════════════════════════════════════════════
vendor_groups = mnm1_ds.get_vendor_groups()
vendor_dice   = {}

for vendor, indices in vendor_groups.items():
    v_slices = [mnm1_slices[i] for i in indices]
    v_ds = DomainShiftDataset(v_slices, transforms=val_tf)
    v_dl = DataLoader(v_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    v_res = run_inference(v_dl)
    dice_per_patient = (
        pd.DataFrame(v_res['dice_records'])['dice_mean']
        .replace([np.inf,-np.inf], np.nan).dropna().tolist()
    )
    vendor_dice[vendor] = dice_per_patient
    mean = np.mean(dice_per_patient) if dice_per_patient else 0
    print(f'  {vendor:<12} n={len(dice_per_patient):4d}  Dice={mean:.4f}')

vendors = sorted(vendor_dice.keys())
data    = [vendor_dice[v] for v in vendors]

fig, ax = plt.subplots(figsize=(max(6, len(vendors)*2), 5))
bp = ax.boxplot(data, labels=vendors, patch_artist=True, notch=False, widths=.5)
palette = sns.color_palette('tab10', len(vendors))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color); patch.set_alpha(.75)
for i, (scores, v) in enumerate(zip(data, vendors)):
    if scores:
        jx = np.random.normal(i+1, .07, len(scores))
        ax.scatter(jx, scores, alpha=.35, s=14, color='grey', zorder=3)
        ax.text(i+1, np.mean(scores)+.02, f'{np.mean(scores):.3f}',
                ha='center', fontsize=9, fontweight='bold')

ax.axhline(0.85, color='red',  ls='--', alpha=.6, lw=1.5, label='DSC=0.85 (clinical)')
ax.axhline(0.90, color='grey', ls='--', alpha=.4, lw=1.2, label='DSC=0.90')

# Cross-vendor gap
means = [np.mean(v) for v in data if v]
delta = max(means) - min(means) if means else 0
ax.set_title(f'M&Ms-1 — Domain Invariance per Scanner Vendor\n'
             f'Cross-vendor gap Δ = {delta:.3f}', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Dice'); ax.set_xlabel('Scanner Vendor')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout()
plt.savefig(OUT_FIG / 'mnm1_domain_invariance.png', dpi=150, bbox_inches='tight')
plt.show()

pd.DataFrame({'vendor': vendors,
              'dice_mean': [np.mean(v) if v else 0 for v in data],
              'dice_std':  [np.std(v)  if v else 0 for v in data],
              'n_slices':  [len(v) for v in data],
             }).to_csv(OUT_EVAL/'mnm1_domain_invariance.csv', index=False)
print('Saved: mnm1_domain_invariance.png')

In [ ]:
# ════════════════════════════════════════════════════
# [13] PLOT — M&Ms-1 ECE + Reliability Diagram
# ════════════════════════════════════════════════════
mnm1_ece, ba, bc, bn = expected_calibration_error(
    mnm1['all_probs'], mnm1['all_labels'], n_bins=ECE_BINS
)
mnm1_aurrc = uncertainty_error_correlation(mnm1['all_unc'], mnm1['all_correct'])

print(f'M&Ms-1 ECE  : {mnm1_ece:.4f}  (ACDC: {ece:.4f}  |  ΔECE = {mnm1_ece-ece:+.4f})')
print(f'M&Ms-1 AURRC: {mnm1_aurrc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reliability diagram
x = np.arange(ECE_BINS)
axes[0].bar(x, ba,  .8, color='#1976D2', alpha=.85, label='Accuracy')
axes[0].bar(x, bc,  .8, color='#E53935', alpha=.30, label='Confidence')
axes[0].plot([0,ECE_BINS-1],[0,1],'k--',alpha=.4,lw=1,label='Perfect')
axes[0].set_title(f'M&Ms-1 Reliability Diagram\nECE = {mnm1_ece:.4f}  '
                  f'(ACDC ECE = {ece:.4f})', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Confidence Bin'); axes[0].set_ylabel('Accuracy / Confidence')
axes[0].set_ylim(0,1); axes[0].legend(); axes[0].grid(alpha=.3)
axes[0].set_xticks(x[::3])
axes[0].set_xticklabels([f'{i/ECE_BINS:.2f}' for i in range(ECE_BINS)][::3])

# Coverage-Accuracy
covs, accs = coverage_accuracy_curve(mnm1['all_unc'], mnm1['all_correct'])
baseline_mnm1 = float(mnm1['all_correct'].mean())
axes[1].plot(covs*100, accs*100, color='#1976D2', lw=2, label='M&Ms-1')
axes[1].axhline(baseline_mnm1*100, color='grey', ls='--', lw=1.5,
                label=f'Baseline ({baseline_mnm1*100:.1f}%)')
axes[1].set_xlabel('Coverage (%)'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('M&Ms-1 — Coverage-Accuracy Curve', fontsize=11, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(OUT_FIG / 'mnm1_reliability_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

pd.DataFrame({'dataset':['MnM1'],'ece':[mnm1_ece],'aurrc':[mnm1_aurrc],
              'acdc_ece':[ece],'delta_ece':[mnm1_ece-ece]
             }).to_csv(OUT_EVAL/'mnm1_calibration.csv', index=False)
print('Saved: mnm1_reliability_diagram.png')

In [ ]:
# ════════════════════════════════════════════════════
# [14] SUMMARY TABLE — tất cả metrics
# ════════════════════════════════════════════════════
rows = []
for dataset, res, ds_ece in [('ACDC', acdc, ece), ('M&Ms-1', mnm1, mnm1_ece)]:
    s = res['summary']
    rows.append({
        'Dataset':  dataset,
        'Dice_RV':  f"{s.get('dice_c1_mean',0):.4f}",
        'Dice_Myo': f"{s.get('dice_c2_mean',0):.4f}",
        'Dice_LV':  f"{s.get('dice_c3_mean',0):.4f}",
        'Dice_Mean':f"{s.get('dice_mean_mean',0):.4f}",
        'HD95_mean':f"{s.get('hd95_mean_mean',0):.2f}",
        'ECE':      f"{ds_ece:.4f}",
    })

summary_df = pd.DataFrame(rows).set_index('Dataset')
summary_df.to_csv(OUT_EVAL / 'full_summary.csv')

print('\n' + '='*60)
print('FULL EVALUATION SUMMARY')
print('='*60)
print(summary_df.to_string())
print('='*60)
print(f'\nAll figures → {OUT_FIG}')
print(f'All CSVs    → {OUT_EVAL}')